In [1]:
#Practica WEB SCRAPING con Python 
#Request sirve para generar la peticion a la pagina web
#beautifulsoup4 sirve para darle formato a los items descargados
#pandas manejo de ficheros
#GET recupera datos de recursos
#POST envia datos al servidor para crear nuevos recursos o procesar como el envio de formularios
#PUT reemplaza completamente un recurso existente con la nueva 
#DELETE para eliminar
#
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

BASE_URL = "https://books.toscrape.com"
print("Librerias cargadas correctamente")
response = requests.get(BASE_URL, timeout=10)
print("Codigo de estado", response.status_code)
print("Longitud del HTML:", len(response.text))
#Vista rapida a los primeros 500 caracteres
print(response.text[:500])

soup = BeautifulSoup(response.text, "html.parser")

#Buscamos todos los elementos <article class= "product_pod">
libros_html = soup.select("article.product_pod")
print(f"Se encontraron {len(libros_html)} libros en esta pagina")
#Veamos el primero
print(libros_html[0].prettify()[:600])

def extraer_rating(clases_css):
    """El rating viene como una clase CSS: 'star_rating Three' -> 3"""
    mapa = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    for clase in clases_css:
        if clase in mapa:
            return mapa[clase]
    return None

libros = []
for item in libros_html:
    titulo = item.h3.a["title"]
    precio_texto = item.select_one("p.price_color").text.strip()
    precio = float(precio_texto.replace("£", "").replace("Â", ""))
    disponibilidad = item.select_one("p.instock.availability").text.strip()
    rating_clases = item.select_one("p.star-rating")["class"] #Ejempo startrating three
    rating = extraer_rating(rating_clases)
    enlace_relativo = item.h3.a["href"]

    libros.append({
        "titulo": titulo,
        "precio_gbp": precio,
        "disponibilidad": disponibilidad,
        "rating": rating,
        "enlace": BASE_URL + "/" + enlace_relativo
    })

df_pagina1 = pd.DataFrame(libros)
print(df_pagina1.head(10))

df_pagina1.to_csv("libros_toscrape_pag1.csv", index=False, encoding="utf-8-sig")
df_pagina1.to_excel("libros_toscrape_pag1.xlsx", index=False)

print("Archivos guardados: libros_toscrape_pag1.csv y libros_toscrape_pag1.xlsx")





Librerias cargadas correctamente


ConnectionError: HTTPSConnectionPool(host='books.toscrape.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='books.toscrape.com', port=443): Failed to resolve 'books.toscrape.com' ([Errno 11001] getaddrinfo failed)"))